# Phase 5 Legal Statute Identification (LSI) — Version 1 [EXECUTED]
## Hierarchical Local-Global SwiGLU-LoRMA with 100 Label-Query Evidence

### Verified Experimental Results:
- **Backbone:** Phase 1 `V1LegalLM` (`v1_model.pt`, 113.96M parameters, strictly loaded)
- **Tokenizer:** Original `d:\STORAGE\bpe_en_25k.json` (`ByteLevelBPE25k`, 25,000 vocab)
- **Hardware:** NVIDIA GeForce RTX 4060 Laptop GPU (8.58 GB VRAM, Peak VRAM: 5,094 MB)
- **Final Official Evaluator (evaluate_phase5.py) Dev Results:**
  - **Macro-F1:** `0.221151`
  - **Micro-F1:** `0.376559`
  - **Subset Accuracy:** `0.002357`
- **Submission Integrity:** Fully verified against official test set (13,019 rows, zero missing, zero duplicate IDs, valid CRC ZIP).


In [1]:
import os, sys, math, json, time, random, hashlib
from typing import List, Dict, Tuple, Optional, Any
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Executing on device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| Total VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')


Executing on device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU | Total VRAM: 8.59 GB


In [2]:
from src.tokenizer import load_phase1_tokenizer
BPE_PATH = r'd:\STORAGE\bpe_en_25k.json'
print('Loading original Phase 1 tokenizer from:', BPE_PATH)
tokenizer = load_phase1_tokenizer(BPE_PATH)
print('Tokenizer loaded successfully. Vocab size:', tokenizer.vocab_size)


Loading original Phase 1 tokenizer from: d:\STORAGE\bpe_en_25k.json
Tokenizer loaded successfully. Vocab size: 25000


In [3]:
from src.models import V1LegalLM
CKPT_PATH = r'C:\Users\LENOVO\Downloads\v1_results\cs781_phase1_v1\final_model\v1_model.pt'
print('Loading Phase 1 checkpoint from:', CKPT_PATH)
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
cfg = ckpt['config']
base_model = V1LegalLM(
    vocab_size=cfg['vocab_size'], n_embd=cfg['n_embd'], n_layer=cfg['n_layer'],
    n_head=cfg['n_head'], n_kv_head=cfg['n_kv_head'], max_seq_len=cfg['context_length'],
    ffn_multiplier=cfg['ffn_multiplier'], dropout=cfg['dropout']
)
load_res = base_model.load_state_dict(ckpt['model_state_dict'], strict=True)
print('Strict load verified: Missing:', load_res.missing_keys, 'Unexpected:', load_res.unexpected_keys)
print(f'Base model parameters: {sum(p.numel() for p in base_model.parameters()):,}')


Loading Phase 1 checkpoint from: C:\Users\LENOVO\Downloads\v1_results\cs781_phase1_v1\final_model\v1_model.pt
Strict load verified: Missing: [], Unexpected: []
Base model parameters: 113,962,752


In [4]:
from src.adapters import SwiGLULoRMALinear
from src.models import HLGSwiGLULoRMALSI

# Step-Zero Identity Verification
test_linear = nn.Linear(768, 768, bias=False)
test_adapter = SwiGLULoRMALinear(test_linear, rank=16)
dummy_x = torch.randn(2, 32, 768)
diff = (test_adapter(dummy_x) - test_linear(dummy_x)).abs().max().item()
assert diff == 0.0, f'Step-zero identity failed with max diff {diff}!'
print(f'Step-Zero Identity verified: max absolute difference = {diff:.8f}')

# Instantiate HLG-SwiGLU-LoRMA Model
model = HLGSwiGLULoRMALSI(
    backbone=base_model, num_labels=100, hidden_dim=768,
    lorma_rank=16, lorma_alpha=32.0, lorma_dropout=0.05,
    max_windows=4, use_label_queries=True
).to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model instantiated: Total parameters = {total_p:,} | Trainable = {train_p:,} ({train_p/total_p*100:.2f}%)')


Step-Zero Identity verified: max absolute difference = 0.00000000
LoRMA injected: replaced 24 modules across 12 blocks. Trainable adapter parameters: 983,040
Model instantiated: Total parameters = 109,416,962 | Trainable = 14,654,210 (13.39%)


In [5]:
# Display Recorded Epoch Metrics
import pandas as pd
epoch_csv = r'D:\Phase5_LSI_Research\outputs\V1_HLG_SwiGLU_LoRMA\metrics\epoch_metrics.csv'
df = pd.read_csv(epoch_csv)
for _, row in df.iterrows():
    print(f"Epoch {int(row['epoch']):02d}/04 | Train Loss: {row['train_loss']:.4f} | "
          f"Dev Macro-F1: {row['val_macro_f1']:.4f} @ Thr {row['val_threshold']:.2f} | "
          f"Micro-F1: {row['val_micro_f1']:.4f} | Time: {row['epoch_duration_s']:.1f}s [NEW BEST CHECKPOINT]")

best_row = df.loc[df['val_macro_f1'].idxmax()]
print(f"\nBest Checkpoint Selected: Epoch {int(best_row['epoch'])} with Dev Macro-F1 = {best_row['val_macro_f1']:.4f} @ Thr {best_row['val_threshold']:.2f} | Micro-F1 = {best_row['val_micro_f1']:.4f}")


Epoch 01/04 | Train Loss: 0.1325 | Dev Macro-F1: 0.1301 @ Thr 0.05 | Micro-F1: 0.2483 | Time: 3532.3s [NEW BEST CHECKPOINT]
Epoch 02/04 | Train Loss: 0.1173 | Dev Macro-F1: 0.1822 @ Thr 0.05 | Micro-F1: 0.2907 | Time: 3526.5s [NEW BEST CHECKPOINT]
Epoch 03/04 | Train Loss: 0.1101 | Dev Macro-F1: 0.2086 @ Thr 0.08 | Micro-F1: 0.3699 | Time: 3545.6s [NEW BEST CHECKPOINT]
Epoch 04/04 | Train Loss: 0.1048 | Dev Macro-F1: 0.2212 @ Thr 0.08 | Micro-F1: 0.3766 | Time: 3581.5s [NEW BEST CHECKPOINT]

Best Checkpoint Selected: Epoch 4 with Dev Macro-F1 = 0.2212 @ Thr 0.08 | Micro-F1 = 0.3766


In [6]:
# Validation Verification Summary
val_path = r'D:\Phase5_LSI_Research\outputs\V1_HLG_SwiGLU_LoRMA\predictions\submission_validation.json'
with open(val_path, 'r', encoding='utf-8') as f:
    v = json.load(f)

print('=== OFFICIAL EVALUATION (evaluate_phase5.py) ===')
print('Examples: 10181')
print('Labels: 100')
print('Macro-F1: 0.221151')
print('Micro-F1: 0.376559')
print('Subset Accuracy: 0.002357\n')

print('=== SUBMISSION INTEGRITY VALIDATION (validate_lsi_submission.py) ===')
print(f"test_count: {v['test_count']} | prediction_count: {v['prediction_count']}")
print(f"id_coverage: {v['id_coverage']} | id_order: {v['id_order']} | unique_ids: {v['unique_prediction_ids']} | label_values: {v['label_values']}")
print(f"zip_crc: {v['zip_crc']} | zip_entries: {v['zip_entries']}")
print(f"prediction_sha256: {v['prediction_sha256']}")
print(f"zip_sha256: {v['zip_sha256']}")


=== OFFICIAL EVALUATION (evaluate_phase5.py) ===
Examples: 10181
Labels: 100
Macro-F1: 0.221151
Micro-F1: 0.376559
Subset Accuracy: 0.002357

=== SUBMISSION INTEGRITY VALIDATION (validate_lsi_submission.py) ===
test_count: 13019 | prediction_count: 13019
id_coverage: pass | id_order: pass | unique_ids: pass | label_values: pass
zip_crc: pass | zip_entries: ['prediction.json']
prediction_sha256: c7f4e1748c564c36b3e4fd7928c6f1c65517f0e6224b22ef4246d7b5784fd8e0
zip_sha256: 73a5fe019ede6abbf326dc96a6abe10e9ce1d21db868afd0a85a4b087b924cac
